# Notebook guide

This notebook compares portfolio allocation methods on pre-selected ETF baskets.

## Inputs
- Selection artifacts from outputs/correlation_analysis
- Shared daily close data exported by data_pipeline

## What to verify
- Weight concentration and effective number of holdings
- Risk/return trade-offs across methods
- Sensitivity to selected basket source

Run cells top-to-bottom to avoid stale state.

# Explore Allocation Methods

This notebook computes portfolio weights for a fixed 10-ETF basket using six allocation strategies.
Each strategy cell calls the corresponding optimizer from `portfolio_allocation/`, applies
per-asset weight bounds, and shows the resulting allocation alongside analytical portfolio statistics
derived from the in-sample covariance matrix.

**Strategies covered:** equal weight (baseline), min variance, max Sharpe, risk parity,
max diversification, HRP, min CVaR.

**No backtest here.** Historical performance evaluation lives in `notebooks/01_project_walkthrough/explore_buy_and_hold.ipynb`.
Take the weights from here and plug them in there.

All return inputs and portfolio statistics use **log returns** throughout:
- ann_return = w^T mean_ret * 252 (annualised log return)
- ann_vol = sqrt(w^T cov w * 252) (annualised log volatility)
- Sharpe = (ann_return - rf) / ann_vol

In [1]:
import pathlib
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / "GUIDE_ROOT.md").exists():
        for _candidate in [_p / "src", _p]:
            if str(_candidate) not in sys.path:
                sys.path.insert(0, str(_candidate))
        PROJECT_ROOT = _p
        break

from correlation_analysis import correlate_utils as utils
import portfolio_allocation as methods
from portfolio_allocation import allocation_utils

TICKERS = [
    "VOO",
    "VEA",
    "VWO",
    "TLT",
    "LQD",
    "IAU",
    "DBA",
    "VNQ",
    "XLP",
    "VWOB",
]

START_DATE = "2018-01-01"
END_DATE = "2025-12-31"
MIN_WEIGHT = 0.05
MAX_WEIGHT = 0.20
RISK_FREE = utils.RISK_FREE

prices_long = utils.load_daily_closes(TICKERS, start_date=START_DATE, end_date=END_DATE)

_stub = pd.DataFrame(
    {
        "ticker": TICKERS,
        "vol_combined": [1.0] * len(TICKERS),
        "start_date": [START_DATE] * len(TICKERS),
    }
)
log_ret, _ = utils.build_return_matrix(prices_long, _stub)
log_ret = log_ret.reindex(columns=TICKERS).dropna(how="all")

final_tickers = log_ret.columns.tolist()
n_assets = len(final_tickers)
print(
    f"Log-return matrix: {log_ret.shape}  ({n_assets} ETFs, {len(log_ret)} daily obs)"
)
print(f"Window: {log_ret.index[0].date()}  to  {log_ret.index[-1].date()}")

filled = log_ret.fillna(log_ret.median())
cov = filled.cov().values
mean_ret = filled.mean().values
returns_arr = filled.values

allocation_utils.validate_weight_bounds(n_assets, MIN_WEIGHT, MAX_WEIGHT)
print(
    f"Weight bounds [{MIN_WEIGHT:.0%}, {MAX_WEIGHT:.0%}] feasible for {n_assets} assets."
)


def portfolio_stats(w, label=""):
    ann_ret = float(w @ mean_ret) * 252
    ann_vol = float(np.sqrt(max(w @ cov @ w * 252, 0.0)))
    sharpe = (ann_ret - RISK_FREE) / ann_vol if ann_vol > 1e-12 else float("nan")
    denom = float(np.sqrt(w @ cov @ w))
    dr = float(w @ np.sqrt(np.diag(cov))) / denom if denom > 1e-14 else 1.0
    eff_n = 1.0 / float(np.square(w).sum()) if np.square(w).sum() > 1e-14 else 0.0
    return {
        "strategy": label,
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "div_ratio": dr,
        "eff_n": eff_n,
    }


all_weights = {}
all_stats = []


def _show(w, label):
    s = portfolio_stats(w, label)
    all_weights[label] = w
    all_stats.append(s)
    print(f"\n--- {label} ---")
    print(
        f"  ann_return={s['ann_return']:.2%}  ann_vol={s['ann_vol']:.2%}  "
        f"Sharpe={s['sharpe']:.2f}  DR={s['div_ratio']:.2f}  eff-N={s['eff_n']:.1f}"
    )
    for t, wi in zip(final_tickers, w):
        print(f"    {t:<6} {wi:.2%}")
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.barh(final_tickers, w * 100, color="#3498db")
    ax.axvline(
        MIN_WEIGHT * 100, color="red", ls="--", alpha=0.6, label=f"min {MIN_WEIGHT:.0%}"
    )
    ax.axvline(
        MAX_WEIGHT * 100,
        color="orange",
        ls="--",
        alpha=0.6,
        label=f"max {MAX_WEIGHT:.0%}",
    )
    ax.axvline(100 / n_assets, color="grey", ls=":", alpha=0.6, label="equal wt")
    ax.set_xlabel("Weight (%)")
    ax.set_title(label.replace("_", " ").title())
    ax.legend(fontsize=8)
    ax.invert_yaxis()
    plt.show()


print("Setup complete.")

Fetching daily closes for 10 symbols (2018-01-01 to 2025-12-31) ...
Fetched 20,110 rows (10 symbols, 2011 unique dates)
Price matrix shape before coverage filter: (2011, 10) (2011 daily periods)
Daily log-return matrix shape: (2010, 10)
Log-return matrix: (2010, 10)  (10 ETFs, 2010 daily obs)
Window: 2018-01-03  to  2025-12-31
Weight bounds [5%, 20%] feasible for 10 assets.
Setup complete.


## Equal Weight (baseline)

Assigns w_i = 1/N. Requires no return or covariance estimates, so it has zero estimation error.
Serves as the benchmark for every other strategy.

In [2]:
w = np.ones(n_assets) / n_assets
w = allocation_utils.apply_weight_bounds(w, MIN_WEIGHT, MAX_WEIGHT)
_show(w, "equal_weight")


--- equal_weight ---
  ann_return=6.00%  ann_vol=10.55%  Sharpe=0.10  DR=1.57  eff-N=10.0
    VOO    10.00%
    VEA    10.00%
    VWO    10.00%
    TLT    10.00%
    LQD    10.00%
    IAU    10.00%
    DBA    10.00%
    VNQ    10.00%
    XLP    10.00%
    VWOB   10.00%


## Minimum Variance

Minimises w^T Sigma w subject to sum(w)=1, w>=0.
Ignores expected returns; concentrates on low-vol, low-correlation assets.

In [3]:
raw = methods.min_variance(cov, max_weight=MAX_WEIGHT)
w = allocation_utils.apply_weight_bounds(raw, MIN_WEIGHT, MAX_WEIGHT)
_show(w, "min_variance")


--- min_variance ---
  ann_return=5.28%  ann_vol=8.79%  Sharpe=0.03  DR=1.71  eff-N=8.4
    VOO    5.00%
    VEA    5.00%
    VWO    5.00%
    TLT    12.04%
    LQD    15.00%
    IAU    10.82%
    DBA    15.00%
    VNQ    5.00%
    XLP    12.14%
    VWOB   15.00%


## Maximum Sharpe

Finds the tangency portfolio: max (w^T mu - rf) / sqrt(w^T Sigma w).
Most return-seeking; most sensitive to estimation error in the mean.

In [4]:
raw = methods.max_sharpe(mean_ret, cov, max_weight=MAX_WEIGHT)
w = allocation_utils.apply_weight_bounds(raw, MIN_WEIGHT, MAX_WEIGHT)
_show(w, "max_sharpe")


--- max_sharpe ---
  ann_return=7.57%  ann_vol=10.64%  Sharpe=0.24  DR=1.56  eff-N=8.3
    VOO    15.00%
    VEA    12.10%
    VWO    5.00%
    TLT    5.00%
    LQD    7.90%
    IAU    15.00%
    DBA    15.00%
    VNQ    5.00%
    XLP    15.00%
    VWOB   5.00%


## Risk Parity

Sets weights so every asset contributes equally to total portfolio risk:
w_i * (Sigma w)_i = sigma_p / N for all i.
Overweights low-vol assets (bonds) vs equal weight.

In [5]:
raw = methods.risk_parity(cov, max_weight=MAX_WEIGHT)
w = allocation_utils.apply_weight_bounds(raw, MIN_WEIGHT, MAX_WEIGHT)
_show(w, "risk_parity")


--- risk_parity ---
  ann_return=5.68%  ann_vol=9.69%  Sharpe=0.07  DR=1.66  eff-N=9.7
    VOO    8.21%
    VEA    8.12%
    VWO    8.26%
    TLT    12.71%
    LQD    11.73%
    IAU    10.86%
    DBA    12.11%
    VNQ    7.75%
    XLP    9.73%
    VWOB   10.52%


## Maximum Diversification

Maximises DR = (w^T sigma) / sqrt(w^T Sigma w) where sigma is per-asset vol.
DR=1 means perfect correlation; higher is better.
Favours low-correlation assets regardless of return.

In [6]:
raw = methods.max_diversification(cov, max_weight=MAX_WEIGHT)
w = allocation_utils.apply_weight_bounds(raw, MIN_WEIGHT, MAX_WEIGHT)
_show(w, "max_diversification")


--- max_diversification ---
  ann_return=5.90%  ann_vol=9.25%  Sharpe=0.10  DR=1.74  eff-N=8.7
    VOO    7.08%
    VEA    5.40%
    VWO    9.37%
    TLT    15.00%
    LQD    8.74%
    IAU    13.69%
    DBA    15.00%
    VNQ    5.00%
    XLP    13.95%
    VWOB   6.77%


## HRP (Hierarchical Risk Parity)

Clusters assets on Spearman correlation distance (Ward linkage), then recursively
bisects the dendrogram allocating weight in proportion to inverse cluster variance.
Does not invert the covariance matrix, so it is well-conditioned on small samples.

In [7]:
raw = methods.hrp_weights(log_ret, max_weight=MAX_WEIGHT)
w = allocation_utils.apply_weight_bounds(raw, MIN_WEIGHT, MAX_WEIGHT)
_show(w, "hrp")


--- hrp ---
  ann_return=5.49%  ann_vol=9.44%  Sharpe=0.05  DR=1.64  eff-N=9.2
    VOO    7.70%
    VEA    7.78%
    VWO    7.36%
    TLT    11.34%
    LQD    15.00%
    IAU    10.71%
    DBA    10.89%
    VNQ    6.40%
    XLP    7.83%
    VWOB   15.00%


## Minimum CVaR

Minimises historical CVaR at the 95% level using the Rockafellar-Uryasev formulation:
  min_{w, zeta}  zeta + 1/(T*(1-alpha)) * sum_t max(-r_{p,t} - zeta, 0)
Tail-loss focused; typically underweights fat-tailed assets (EM equity, commodities).

In [8]:
raw = methods.min_cvar(returns_arr, alpha=0.95, max_weight=MAX_WEIGHT)
w = allocation_utils.apply_weight_bounds(raw, MIN_WEIGHT, MAX_WEIGHT)
_show(w, "min_cvar")


--- min_cvar ---
  ann_return=5.34%  ann_vol=8.75%  Sharpe=0.04  DR=1.72  eff-N=8.4
    VOO    5.00%
    VEA    5.00%
    VWO    5.00%
    TLT    12.39%
    LQD    15.00%
    IAU    11.95%
    DBA    15.00%
    VNQ    5.00%
    XLP    10.66%
    VWOB   15.00%


## Cross-Strategy Summary

Analytical statistics (from in-sample mean + covariance, NOT a historical backtest)
and a weight heatmap comparing all strategies.

In [9]:
summary = pd.DataFrame(all_stats)
print("Analytical Portfolio Statistics (log-return based):")
for _, row in summary.iterrows():
    print(
        f"  {row['strategy']:<22} return={row['ann_return']:.2%}  vol={row['ann_vol']:.2%}  "
        f"Sharpe={row['sharpe']:.2f}  DR={row['div_ratio']:.2f}  eff-N={row['eff_n']:.1f}"
    )

weights_df = pd.DataFrame(all_weights, index=final_tickers) * 100
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

im = axes[0].imshow(
    weights_df.values, cmap="Blues", vmin=0, vmax=MAX_WEIGHT * 100, aspect="auto"
)
axes[0].set_xticks(range(len(weights_df.columns)))
axes[0].set_xticklabels(
    [c.replace("_", " ").title() for c in weights_df.columns],
    rotation=25,
    ha="right",
    fontsize=8,
)
axes[0].set_yticks(range(n_assets))
axes[0].set_yticklabels(final_tickers, fontsize=8)
for i in range(n_assets):
    for j in range(len(weights_df.columns)):
        v = weights_df.values[i, j]
        axes[0].text(
            j,
            i,
            f"{v:.1f}%",
            ha="center",
            va="center",
            fontsize=7,
            color="white" if v > 14 else "black",
        )
fig.colorbar(im, ax=axes[0], label="Weight (%)")
axes[0].set_title("Weights by Strategy (%)")

ew = 100.0 / n_assets
dev_df = weights_df - ew
im2 = axes[1].imshow(dev_df.values, cmap="RdBu", vmin=-ew, vmax=ew, aspect="auto")
axes[1].set_xticks(range(len(dev_df.columns)))
axes[1].set_xticklabels(
    [c.replace("_", " ").title() for c in dev_df.columns],
    rotation=25,
    ha="right",
    fontsize=8,
)
axes[1].set_yticks(range(n_assets))
axes[1].set_yticklabels(final_tickers, fontsize=8)
for i in range(n_assets):
    for j in range(len(dev_df.columns)):
        v = dev_df.values[i, j]
        axes[1].text(
            j,
            i,
            f"{v:+.1f}%",
            ha="center",
            va="center",
            fontsize=7,
            color="white" if abs(v) > 5 else "black",
        )
fig.colorbar(im2, ax=axes[1], label="vs equal weight (%)")
axes[1].set_title("Deviation from Equal Weight (blue=over, red=under)")
plt.show()

print("\nTo backtest any allocation, copy TICKERS + WEIGHTS into")
print(
    "notebooks/01_project_walkthrough/explore_buy_and_hold.ipynb and run that notebook."
)

Analytical Portfolio Statistics (log-return based):
  equal_weight           return=6.00%  vol=10.55%  Sharpe=0.10  DR=1.57  eff-N=10.0
  min_variance           return=5.28%  vol=8.79%  Sharpe=0.03  DR=1.71  eff-N=8.4
  max_sharpe             return=7.57%  vol=10.64%  Sharpe=0.24  DR=1.56  eff-N=8.3
  risk_parity            return=5.68%  vol=9.69%  Sharpe=0.07  DR=1.66  eff-N=9.7
  max_diversification    return=5.90%  vol=9.25%  Sharpe=0.10  DR=1.74  eff-N=8.7
  hrp                    return=5.49%  vol=9.44%  Sharpe=0.05  DR=1.64  eff-N=9.2
  min_cvar               return=5.34%  vol=8.75%  Sharpe=0.04  DR=1.72  eff-N=8.4



To backtest any allocation, copy TICKERS + WEIGHTS into
notebooks/01_project_walkthrough/explore_buy_and_hold.ipynb and run that notebook.
